In [2]:
import requests
import re
import json
import pandas as pd

URL = "https://rft.gg/player/16-zeus?year=2025"

# ============================================================
# 1. SEITE LADEN
# ============================================================

response = requests.get(
    URL,
    headers={"User-Agent": "Mozilla/5.0"}
)

print("Status:", response.status_code)

html = response.text
print("HTML Länge:", len(html))

# ============================================================
# 2. ALLE __next_f PUSH DATEN SAMMELN
# ============================================================

matches = re.findall(
    r'self\.__next_f\.push\(\[1,"(.*?)"\]\)',
    html,
    re.DOTALL
)

print("__next_f Blöcke gefunden:", len(matches))

# ============================================================
# 3. ESCAPED CONTENT ZURÜCK IN NORMALEN TEXT UMWANDELN
# ============================================================

blocks = []

for block in matches:
    try:
        # JSON-String dekodieren
        decoded = json.loads('"' + block + '"')
        blocks.append(decoded)
    except:
        blocks.append(block)

# ============================================================
# 4. RELEVANTE BLÖCKE AUTOMATISCH FINDEN
# ============================================================

print("\nRelevante Blöcke:")

for i, block in enumerate(blocks):

    score = 0

    for keyword in [
        '"eventId"',
        '"eventName"',
        '"gamesPlayed"',
        '"wins"',
        '"losses"',
        '"dpm"',
        '"rft"',
        '"kda"'
    ]:
        if keyword in block:
            score += 1

    if score >= 3:
        print(
            f"Block {i:3} | "
            f"Score {score} | "
            f"Länge {len(block)}"
        )

# ============================================================
# 5. ALLE JSON-OBJEKTE MIT eventName SUCHEN
# ============================================================

print("\n" + "=" * 80)
print("EVENTS")
print("=" * 80)

event_pattern = re.compile(
    r'\{"eventId":(\-?\d+),"eventName":"([^"]+)"'
)

events = event_pattern.findall("\n".join(blocks))

for event_id, event_name in events:
    print(event_id, "|", event_name)

# ============================================================
# 6. NACH PERFORMANCE-DATEN SUCHEN
# ============================================================

print("\n" + "=" * 80)
print("PERFORMANCE-DATEN")
print("=" * 80)

all_text = "\n".join(blocks)

keywords = [
    "performances",
    "dpm",
    "rft",
    "KDA",
    "kda"
]

for keyword in keywords:
    print(
        f"{keyword:15}:",
        all_text.find(keyword)
    )

# ============================================================
# 7. RELEVANTE STELLEN UM dpm AUSGEBEN
# ============================================================

positions = [
    m.start()
    for m in re.finditer("dpm", all_text, re.IGNORECASE)
]

print("\nDPM-Treffer:", len(positions))

for pos in positions[:20]:

    start = max(0, pos - 500)
    end = min(len(all_text), pos + 1000)

    print("\n" + "-" * 80)
    print(all_text[start:end])

Status: 200
HTML Länge: 652872
__next_f Blöcke gefunden: 15

Relevante Blöcke:
Block  10 | Score 4 | Länge 5855

EVENTS
1852 | KeSPA Cup 2026
200 | Esports World Cup 2026
167 | MSI 2026
157 | LCK 2026 Road to MSI
246 | Esports World Cup 2026 Korea
1856 | LCK 2026 Season
74 | LCK 2026 Cup
2055 | KeSPA Cup 2025
1605 | World Championship 2025
1603 | Esports World Cup 2025
1536 | LCK 2025 Road to MSI
1538 | LCK 2025 Season
1587 | First Stand 2025
1534 | LCK 2025 Cup
1465 | World Championship 2024
1428 | LCK 2024 Regional Finals
1464 | Esports World Cup 2024
1427 | LCK 2024 Summer
1462 | MSI 2024
1426 | LCK 2024 Spring
1332 | World Championship 2023
2671 | Asian Games 2022
1265 | LCK 2023 Summer
1317 | MSI 2023
1264 | LCK 2023 Spring
1200 | World Championship 2022
1126 | LCK 2022 Summer
1195 | MSI 2022
1125 | LCK 2022 Spring
991 | LCK 2021 Spring
-2020 | 2020
-2019 | 2019
-2014 | 2014
-2013 | 2013
-2012 | 2012

PERFORMANCE-DATEN
performances   : 116814
dpm            : 21880
rft            

In [3]:
import requests
import re
import json
from bs4 import BeautifulSoup

# ============================================================
# 1. URL
# ============================================================

URL = "https://rft.gg/player/16-zeus?year=2025"

response = requests.get(URL)

print("Status:", response.status_code)
print("HTML Länge:", len(response.text))

html = response.text
soup = BeautifulSoup(html, "html.parser")


# ============================================================
# 2. ALLE __next_f BLÖCKE
# ============================================================

blocks = []

for script in soup.find_all("script"):
    text = script.string or script.get_text()

    if text and "self.__next_f.push" in text:
        blocks.append(text)

print("__next_f Blöcke gefunden:", len(blocks))


# ============================================================
# 3. PERFORMANCE-BLOCK FINDEN
# ============================================================

performance_block = None

for i, block in enumerate(blocks):

    if '"performances":' in block and '"gameId":' in block:

        performance_block = block

        print()
        print("Performance-Block gefunden!")
        print("Block:", i)
        print("Länge:", len(block))

        break


if performance_block is None:

    print()
    print("Kein Performance-Block gefunden.")

    print()
    print("Blöcke mit 'performances':")

    for i, block in enumerate(blocks):
        if "performances" in block:
            print(
                "Block", i,
                "| Länge", len(block),
                "| gameId:", "gameId" in block
            )

    raise Exception("Performance-Block nicht gefunden.")


# ============================================================
# 4. PERFORMANCES-ARRAY BEREICH FINDEN
# ============================================================

marker = '"performances":['

start = performance_block.find(marker)

if start == -1:
    raise Exception("performances-Array nicht gefunden.")

array_start = start + len(marker)

print()
print("performances gefunden bei:", start)


# ============================================================
# 5. ARRAY BIS ZUM PASSENDEN ] VERFOLGEN
# ============================================================

depth = 1
end = array_start

in_string = False
escape = False

while end < len(performance_block):

    char = performance_block[end]

    if in_string:

        if escape:
            escape = False

        elif char == "\\":
            escape = True

        elif char == '"':
            in_string = False

    else:

        if char == '"':
            in_string = True

        elif char == "[":
            depth += 1

        elif char == "]":
            depth -= 1

            if depth == 0:
                break

    end += 1


if depth != 0:
    raise Exception("Ende des performances-Arrays nicht gefunden.")


performance_array_text = performance_block[array_start:end]


print("Performance-Array Länge:", len(performance_array_text))


# ============================================================
# 6. EINZELNE PERFORMANCE-OBJEKTE EXTRAHIEREN
# ============================================================

pattern = r'\{'
objects = []

pos = 0

while pos < len(performance_array_text):

    match = re.search(r'\{', performance_array_text[pos:])

    if not match:
        break

    obj_start = pos + match.start()

    depth = 1
    i = obj_start + 1

    in_string = False
    escape = False

    while i < len(performance_array_text):

        char = performance_array_text[i]

        if in_string:

            if escape:
                escape = False

            elif char == "\\":
                escape = True

            elif char == '"':
                in_string = False

        else:

            if char == '"':
                in_string = True

            elif char == "{":
                depth += 1

            elif char == "}":
                depth -= 1

                if depth == 0:
                    break

        i += 1

    obj_text = performance_array_text[obj_start:i + 1]

    try:
        obj = json.loads(obj_text)

        if "gameId" in obj:
            objects.append(obj)

    except json.JSONDecodeError:
        pass

    pos = i + 1


# ============================================================
# 7. ERGEBNIS
# ============================================================

print()
print("=" * 80)
print("PERFORMANCE-DATEN")
print("=" * 80)

print("Games gefunden:", len(objects))

if objects:

    print()
    print("Erstes Game:")
    print(json.dumps(objects[0], indent=2))

    print()
    print("Letztes Game:")
    print(json.dumps(objects[-1], indent=2))


# ============================================================
# 8. KURZE ÜBERSICHT
# ============================================================

print()
print("=" * 80)
print("ÜBERSICHT")
print("=" * 80)

for game in objects[:10]:

    print(
        game.get("gameId"),
        "|",
        game.get("gameDate"),
        "|",
        game.get("rftRating"),
        "| KDA:", game.get("kda"),
        "| DPM:", game.get("dpm"),
        "| Win:", game.get("win"),
        "| Role:", game.get("role")
    )

Status: 200
HTML Länge: 652744
__next_f Blöcke gefunden: 12

Kein Performance-Block gefunden.

Blöcke mit 'performances':
Block 10 | Länge 92027 | gameId: True


Exception: Performance-Block nicht gefunden.

In [4]:
# ALLE Blöcke finden, die tatsächlich "performances" enthalten

for i, block in enumerate(blocks):
    if "performances" in block:
        pos = block.find("performances")

        print("=" * 80)
        print("BLOCK:", i)
        print("LÄNGE:", len(block))
        print("POSITION:", pos)
        print()
        print(repr(block[max(0, pos-100):pos+300]))

BLOCK: 10
LÄNGE: 92027
POSITION: 1916

'dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\\"],\\"RecentFormTimeline\\"]\\n5b:[\\"$\\",\\"$L64\\",null,{\\"performances\\":[{\\"gameId\\":5913248,\\"matchId\\":14015,\\"matchSlug\\":\\"gen.g-esports-vs-hanwha-life-esports-2025-01-17\\",\\"gameDate\\":\\"$D2025-01-17T08:08:00.000Z\\",\\"opponentTeamId\\":8768,\\"opponentTeamName\\":\\"Gen.G Esports\\",\\"opponentTeamShortName\\":\\"GEN\\",\\"opponentTeamSlug\\":\\"gen.g-esports\\",'


In [5]:
# ALLE Blöcke finden, die tatsächlich "performances" enthalten

for i, block in enumerate(blocks):
    if "performances" in block:
        pos = block.find("performances")

        print("=" * 80)
        print("BLOCK:", i)
        print("LÄNGE:", len(block))
        print("POSITION:", pos)
        print()
        print(repr(block[max(0, pos-100):pos+300]))

BLOCK: 10
LÄNGE: 92027
POSITION: 1916

'dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\\"],\\"RecentFormTimeline\\"]\\n5b:[\\"$\\",\\"$L64\\",null,{\\"performances\\":[{\\"gameId\\":5913248,\\"matchId\\":14015,\\"matchSlug\\":\\"gen.g-esports-vs-hanwha-life-esports-2025-01-17\\",\\"gameDate\\":\\"$D2025-01-17T08:08:00.000Z\\",\\"opponentTeamId\\":8768,\\"opponentTeamName\\":\\"Gen.G Esports\\",\\"opponentTeamShortName\\":\\"GEN\\",\\"opponentTeamSlug\\":\\"gen.g-esports\\",'


In [6]:
import json

block = blocks[13]

# Escaping entfernen
decoded = block.replace('\\"', '"')

# Jetzt sollte die Struktur normal aussehen
pos = decoded.find('"performances":')

print("Position:", pos)
print(decoded[pos:pos+500])

IndexError: list index out of range

In [7]:
import json

block = blocks[13]

# 1. Escaping entfernen
decoded = block.replace('\\"', '"')

# 2. performances finden
marker = '"performances":['
start = decoded.find(marker)

print("performances Position:", start)

if start == -1:
    raise Exception("performances nicht gefunden!")

# Position direkt nach [
array_start = start + len(marker)

# 3. Ende des Arrays finden
depth = 1
i = array_start

while i < len(decoded) and depth > 0:

    if decoded[i] == "[":
        depth += 1

    elif decoded[i] == "]":
        depth -= 1

    i += 1

array_text = decoded[array_start:i-1]

print("Array-Länge:", len(array_text))


# 4. Einzelne Game-Objekte finden
games = []

decoder = json.JSONDecoder()
pos = 0

while pos < len(array_text):

    # Leerzeichen/Kommas überspringen
    while pos < len(array_text) and array_text[pos] in " ,\n":
        pos += 1

    if pos >= len(array_text):
        break

    try:
        obj, consumed = decoder.raw_decode(array_text[pos:])

        if "gameId" in obj:
            games.append(obj)

        pos += consumed

    except json.JSONDecodeError as e:
        print("JSON-Fehler bei Position:", pos)
        print(array_text[pos:pos+200])
        break


# 5. Ergebnis
print()
print("=" * 80)
print("GAMES GEFUNDEN:", len(games))
print("=" * 80)

if games:
    print()
    print("Erstes Game:")
    print(json.dumps(games[0], indent=2))

    print()
    print("Letztes Game:")
    print(json.dumps(games[-1], indent=2))

IndexError: list index out of range

In [8]:
import requests
from bs4 import BeautifulSoup

URL = "https://rft.gg/match/14015-gen.g-esports-vs-hanwha-life-esports-2025-01-17"

html_match = requests.get(URL).text

print("Status:", requests.get(URL).status_code)
print("HTML Länge:", len(html_match))

# Alle Stellen mit eventId / eventName anzeigen
for term in ["eventId", "eventName", "LCK 2025", "LCK 2025 Season"]:
    print("\n" + "=" * 80)
    print("SUCHBEGRIFF:", term)
    print("Treffer:", html_match.count(term))

    pos = 0
    for i in range(min(3, html_match.count(term))):
        pos = html_match.find(term, pos)

        print("\n--- Treffer", i + 1, "---")
        print(html_match[max(0, pos-500):pos+1000])

        pos += len(term)

Status: 200
HTML Länge: 1547626

SUCHBEGRIFF: eventId
Treffer: 33

--- Treffer 1 ---
ks/3--scy45zdg78.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/0_g_vaqi14l37.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/0lcf-wgup25mr.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\"],\"MatchListClient\"]\n1b:63049\n1a:63019\n30:[\"$\",\"$L4a\",null,{\"initialMatches\":[],\"initialRecentResults\":[{\"match\":{\"id\":83967,\"slug\":\"team-liquid-vs-flyquest-2026-09-20\",\"scheduledDate\":\"$D2026-09-20T20:00:00.000Z\",\"league\":\"LCS\",\"eventId\":28,\"eventImageUrl\":\"https://uploads.rft.gg/images/esport/tournaments/lcs-2026-summer.png\",\"numberOfGames\":5},\"team1\":{\"id\":296,\"name\":\"Team Liquid\",\"shortName\":\"TLAW\",\"slug\":\"team-liquid\",\"imageUrl\":\"https://uploads.rft.gg/images/esport/teams/team-liquid-2026.png\",\"wins\":3},\"team2\":{\"id\":303,\"name\":\"FlyQuest\",\"shortName\":\"FLY\",\"slug\":\"flyquest\",\"imageUrl\":\

In [9]:
term = '\\"id\\":14015'

pos = html_match.find(term)

print("Position:", pos)
print()
print(html_match[pos-1000:pos+2500])

Position: 267183

nks/0_g_vaqi14l37.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/0lcf-wgup25mr.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/1yxfx86bntb5r.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/2ovhp_n0q1-54.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/0va3ebynjph1x.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/3mz0tn2kqnzd2.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/3n_y2h7gi3aik.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/1pb7rsa9r6ewb.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/2tk2d4fi7ckdh.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/1zsp0q5-j2hw5.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\",\"/_next/static/chunks/1z16il0llv55n.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b\"],\"FloatingCommentCTA\"]\n62:[\"$\",\"$L65\"

In [10]:
# ============================================================
# EVENT-DATEN IN DER NÄHE DES MATCH-OBJEKTS SUCHEN
# ============================================================

match_pos = html.find('"id":14015')

print("Match-Position:", match_pos)

# größeren Bereich um das Match anzeigen
start = max(0, match_pos - 5000)
end = min(len(html), match_pos + 15000)

snippet = html[start:end]

print("\n" + "=" * 100)
print("AUSSCHNITT UM MATCH 14015")
print("=" * 100)

print(snippet)

Match-Position: -1

AUSSCHNITT UM MATCH 14015
<!DOCTYPE html><html data-dpl-id="a7093ce8df295522e7f6508ac7b3d7328c91dd0b" lang="en"><head><meta charSet="utf-8"/><meta name="viewport" content="width=device-width, initial-scale=1, viewport-fit=cover"/><link rel="stylesheet" href="/_next/static/chunks/10gazu5jp5vwr.css?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b" data-precedence="next"/><link rel="stylesheet" href="/_next/static/chunks/0lb7wxgj1jkpy.css?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b" data-precedence="next"/><link rel="preload" as="script" fetchPriority="low" href="/_next/static/chunks/0tmlqphq0ty9t.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b"/><script src="/_next/static/chunks/2kn9i7mj_95ql.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b" async=""></script><script src="/_next/static/chunks/0p35_9ceybk7g.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b" async=""></script><script src="/_next/static/chunks/098mjmi9uga5b.js?dpl=a7093ce8df295522e7f6508ac7b3d7328c91dd0b" asyn

In [1]:
import requests
import re
import json
import time
from collections import OrderedDict

# ============================================================
# 1. PLAYER-SEITE ABRUFEN
# ============================================================

url = "https://rft.gg/player/16-zeus"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status:", response.status_code)
print("HTML Länge:", len(response.text))

html = response.text


# ============================================================
# 2. PERFORMANCE-BLOCK FINDEN
# ============================================================

marker = '\\"performances\\":['

position = html.find(marker)

print("Performances Position:", position)

if position == -1:
    raise Exception("Performances-Array nicht gefunden.")


# ============================================================
# 3. ARRAY AUS DEM HTML HERAUSHOLEN
# ============================================================

start = position + len('\\"performances\\":')

depth = 0
in_string = False
escape = False
end = None

for i in range(start, len(html)):

    c = html[i]

    if escape:
        escape = False
        continue

    if c == "\\":
        escape = True
        continue

    if c == '"':
        in_string = not in_string
        continue

    if in_string:
        continue

    if c == "[":
        depth += 1

    elif c == "]":
        depth -= 1

        if depth == 0:
            end = i + 1
            break

if end is None:
    raise Exception("Ende des Performances-Arrays nicht gefunden.")

array_text = html[start:end]

print("Array-Länge:", len(array_text))


# ============================================================
# 4. ESCAPING ENTFERNEN
# ============================================================

array_text = array_text.replace('\\"', '"')


# ============================================================
# 5. GAMES EXTRAHIEREN
# ============================================================

games = json.loads(array_text)

print()
print("=" * 80)
print("GAMES GEFUNDEN:", len(games))
print("=" * 80)

print("\nErstes Game:")
print(json.dumps(games[0], indent=2, ensure_ascii=False))

print("\nLetztes Game:")
print(json.dumps(games[-1], indent=2, ensure_ascii=False))


# ============================================================
# 6. EINDEUTIGE MATCH-IDS
# ============================================================

match_ids = list(OrderedDict.fromkeys(
    game["matchId"] for game in games
))

print()
print("=" * 80)
print("UNIQUE MATCH IDs:", len(match_ids))
print("=" * 80)


# ============================================================
# 7. MATCH-SEITEN ABRUFEN
# ============================================================

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

results = {}

for i, match_id in enumerate(match_ids, 1):

    # Wir kennen die Slugs bereits aus den Games.
    # Deshalb suchen wir das passende Game.
    game = next(
        g for g in games
        if g["matchId"] == match_id
    )

    slug = game["matchSlug"]

    match_url = f"https://rft.gg/match/{match_id}-{slug}"

    try:
        r = session.get(match_url, timeout=20)

        if r.status_code != 200:
            print(
                f"[{i}/{len(match_ids)}] "
                f"{match_id} -> HTTP {r.status_code}"
            )
            continue

        match_html = r.text

        # ----------------------------------------------------
        # eventId
        # ----------------------------------------------------

        event_ids = re.findall(
            r'\\"eventId\\":(\d+)',
            match_html
        )

        if not event_ids:
            event_ids = re.findall(
                r'"eventId":(\d+)',
                match_html
            )

        event_id = event_ids[0] if event_ids else None

        # ----------------------------------------------------
        # Titel
        # ----------------------------------------------------

        title_match = re.search(
            r'<title>(.*?)</title>',
            match_html,
            re.DOTALL
        )

        title = (
            title_match.group(1).strip()
            if title_match
            else None
        )

        results[match_id] = {
            "matchId": match_id,
            "slug": slug,
            "eventId": event_id,
            "title": title
        }

        print(
            f"[{i}/{len(match_ids)}] "
            f"{match_id} -> eventId={event_id}"
        )

        time.sleep(0.15)

    except Exception as e:

        print(
            f"[{i}/{len(match_ids)}] "
            f"{match_id} -> ERROR: {e}"
        )


# ============================================================
# 8. SPEICHERN
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print("Matches verarbeitet:", len(results))
print("Datei: match_events.json")

Status: 200
HTML Länge: 598514
Performances Position: 511757
Array-Länge: 75459

GAMES GEFUNDEN: 143

Erstes Game:
{
  "gameId": 340,
  "matchId": 9503,
  "matchSlug": "hanwha-life-esports-vs-t1-2026-01-16",
  "gameDate": "$D2026-01-16T10:08:00.000Z",
  "opponentTeamId": 272,
  "opponentTeamName": "T1",
  "opponentTeamShortName": "T1",
  "opponentTeamSlug": "t1",
  "opponentTeamImageUrl": "https://uploads.rft.gg/images/esport/teams/t1.webp",
  "rftRating": 62.8,
  "kda": 2,
  "dpm": 611.2740285435754,
  "gpm": 370.91801766633324,
  "csm": 7.45578231292517,
  "vision": 1.6856731453895633,
  "win": true,
  "role": "TOP"
}

Letztes Game:
{
  "gameId": 5942269,
  "matchId": 69377,
  "matchSlug": "gen.g-esports-vs-hanwha-life-esports-2026-09-13",
  "gameDate": "$D2026-09-13T08:19:00.000Z",
  "opponentTeamId": 8768,
  "opponentTeamName": "Gen.G Esports",
  "opponentTeamShortName": "GEN",
  "opponentTeamSlug": "gen.g-esports",
  "opponentTeamImageUrl": "https://uploads.rft.gg/images/esport/te

In [4]:
import json

# ============================================================
# MATCH -> EVENT ZUORDNUNG
# ============================================================

with open("match_events.json", "r", encoding="utf-8") as f:
    match_events = json.load(f)

# Event aus dem Titel extrahieren
def get_event(title):
    if "LCK 2026 Cup" in title:
        return "LCK 2026 Cup"
    elif "LCK 2026 Season" in title:
        return "LCK 2026 Season"
    elif "LCK 2026 Road to MSI" in title:
        return "LCK 2026 Road to MSI"
    elif "MSI 2026" in title:
        return "MSI 2026"
    elif "Esports World Cup 2026" in title:
        return "Esports World Cup 2026"
    elif "KeSPA Cup 2026" in title:
        return "KeSPA Cup 2026"
    else:
        return "UNKNOWN"


# Jedes Game einem Event zuordnen
games_with_events = []

for game in games:
    match_id = str(game["matchId"])

    if match_id not in match_events:
        print(f"WARNUNG: Match {match_id} nicht in match_events.json")
        continue

    title = match_events[match_id]["title"]
    event = get_event(title)

    game_copy = game.copy()
    game_copy["event"] = event

    games_with_events.append(game_copy)


# ============================================================
# AUSGABE
# ============================================================

print("=" * 80)
print(f"GAMES MIT EVENT: {len(games_with_events)}")
print("=" * 80)

from collections import Counter

event_counts = Counter(game["event"] for game in games_with_events)

for event, count in event_counts.items():
    print(f"{event:<30} {count} Games")


print("\n" + "=" * 80)
print("BEISPIEL")
print("=" * 80)

print(json.dumps(games_with_events[0], indent=2, ensure_ascii=False))


# ============================================================
# SPEICHERN
# ============================================================

with open("games_with_events.json", "w", encoding="utf-8") as f:
    json.dump(games_with_events, f, indent=2, ensure_ascii=False)

print("\nGespeichert als: games_with_events.json")

GAMES MIT EVENT: 143
LCK 2026 Cup                   14 Games
LCK 2026 Season                81 Games
Esports World Cup 2026         16 Games
LCK 2026 Road to MSI           4 Games
MSI 2026                       20 Games
KeSPA Cup 2026                 8 Games

BEISPIEL
{
  "gameId": 340,
  "matchId": 9503,
  "matchSlug": "hanwha-life-esports-vs-t1-2026-01-16",
  "gameDate": "$D2026-01-16T10:08:00.000Z",
  "opponentTeamId": 272,
  "opponentTeamName": "T1",
  "opponentTeamShortName": "T1",
  "opponentTeamSlug": "t1",
  "opponentTeamImageUrl": "https://uploads.rft.gg/images/esport/teams/t1.webp",
  "rftRating": 62.8,
  "kda": 2,
  "dpm": 611.2740285435754,
  "gpm": 370.91801766633324,
  "csm": 7.45578231292517,
  "vision": 1.6856731453895633,
  "win": true,
  "role": "TOP",
  "event": "LCK 2026 Cup"
}

Gespeichert als: games_with_events.json


In [14]:
import json

# ============================================================
# 1. DATEN LADEN
# ============================================================

with open("games_with_events.json", "r", encoding="utf-8") as f:
    games = json.load(f)

# ============================================================
# 2. RELEVANTE EVENTS
# ============================================================

allowed_events = {
    "LCK 2026 Cup",
    "LCK 2026 Season",
    "LCK 2026 Road to MSI",
    "MSI 2026"
}

filtered_games = [
    game for game in games
    if game.get("event") in allowed_events
]

# ============================================================
# 3. AUSGABE
# ============================================================

print("=" * 80)
print(f"RELEVANTE GAMES: {len(filtered_games)}")
print("=" * 80)

from collections import Counter

event_counts = Counter(game["event"] for game in filtered_games)

for event, count in event_counts.items():
    print(f"{event:<30} {count} Games")

# ============================================================
# 4. SPEICHERN
# ============================================================

with open("relevant_games.json", "w", encoding="utf-8") as f:
    json.dump(filtered_games, f, indent=2, ensure_ascii=False)

print()
print("=" * 80)
print("Gespeichert als: relevant_games.json")
print("=" * 80)

RELEVANTE GAMES: 119
LCK 2026 Cup                   14 Games
LCK 2026 Season                81 Games
LCK 2026 Road to MSI           4 Games
MSI 2026                       20 Games

Gespeichert als: relevant_games.json


In [3]:
import requests
import re

url = "https://rft.gg/player/16-zeus"

r = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

html = r.text

print("Status:", r.status_code)
print("HTML Länge:", len(html))

# Suche nach Stellen, an denen Jahreswerte vorkommen
for year in ["2022", "2023", "2024", "2025", "2026"]:
    positions = [m.start() for m in re.finditer(year, html)]

    print()
    print("=" * 80)
    print(year, "->", len(positions), "Treffer")

    for pos in positions[:5]:
        print("\nPOSITION:", pos)
        print(repr(html[max(0, pos-300):pos+500]))

Status: 200
HTML Länge: 598170

2022 -> 21 Treffer

POSITION: 123624
'e\\":\\"World Championship 2023\\",\\"eventYear\\":2023,\\"eventLeague\\":\\"Worlds\\",\\"eventSplit\\":null,\\"startDate\\":\\"$D2023-10-10T00:00:00.000Z\\",\\"eventImageUrl\\":\\"https://uploads.rft.gg/images/esport/tournaments/world-championship-2023-main-event.png\\"},{\\"eventId\\":2671,\\"eventName\\":\\"Asian Games 2022\\",\\"eventYear\\":2023,\\"eventLeague\\":\\"AG\\",\\"eventSplit\\":null,\\"startDate\\":\\"$D2023-09-11T00:00:00.000Z\\",\\"eventImageUrl\\":\\"https://uploads.rft.gg/images/esport/tournaments/asian-games-2022.png\\"},{\\"eventId\\":1265,\\"eventName\\":\\"LCK 2023 Summer\\",\\"eventYear\\":2023,\\"eventLeague\\":\\"LCK\\",\\"eventSplit\\":\\"Summer\\",\\"startDate\\":\\"$D2023-06-07T00:00:00.000Z\\",\\"eventImageUrl\\":\\"https://uploads.rft.gg/images/esport/tournaments/lck-2023-summer.png\\"},{\\"eventId\\":1317,\\"eventName\\":\\"M'

POSITION: 123819
'port/tournaments/world-championship-202

In [6]:
import json

# ============================================================
# 1. GAMES LADEN
# ============================================================

with open("games_with_events.json", "r", encoding="utf-8") as f:
    games = json.load(f)


# ============================================================
# 2. ERLAUBTE EVENTS
# ============================================================

def is_allowed_event(event):

    event = event.lower()

    # LCK
    if "lck" in event:
        return True

    # Internationale Turniere
    if "msi" in event:
        return True

    if "worlds" in event:
        return True

    if "first stand" in event:
        return True

    return False


# ============================================================
# 3. FILTERN
# ============================================================

filtered_games = [
    game
    for game in games
    if is_allowed_event(game.get("event", ""))
]


# ============================================================
# 4. AUSGABE
# ============================================================

from collections import Counter

print("=" * 80)
print("ERLAUBTE GAMES:", len(filtered_games))
print("=" * 80)

counts = Counter(game["event"] for game in filtered_games)

for event, count in counts.items():
    print(f"{event:<40} {count} Games")


print()
print("=" * 80)
print("AUSGESCHLOSSENE EVENTS")
print("=" * 80)

excluded = Counter(
    game["event"]
    for game in games
    if not is_allowed_event(game.get("event", ""))
)

for event, count in excluded.items():
    print(f"{event:<40} {count} Games")


# ============================================================
# 5. SPEICHERN
# ============================================================

with open("filtered_games.json", "w", encoding="utf-8") as f:
    json.dump(
        filtered_games,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("Gespeichert als: filtered_games.json")

ERLAUBTE GAMES: 119
LCK 2026 Cup                             14 Games
LCK 2026 Season                          81 Games
LCK 2026 Road to MSI                     4 Games
MSI 2026                                 20 Games

AUSGESCHLOSSENE EVENTS
Esports World Cup 2026                   16 Games
KeSPA Cup 2026                           8 Games

Gespeichert als: filtered_games.json


In [9]:
import requests
import json
import time
import random

# ============================================================
# DATEIEN LADEN
# ============================================================

with open("filtered_games.json", "r", encoding="utf-8") as f:
    games = json.load(f)

# Bereits verarbeitete Ergebnisse laden, falls vorhanden
try:
    with open("match_events.json", "r", encoding="utf-8") as f:
        results = json.load(f)
except FileNotFoundError:
    results = {}

# ============================================================
# EINDEUTIGE MATCH-IDS
# ============================================================

match_ids = []
seen = set()

for game in games:
    match_id = str(game["matchId"])

    if match_id not in seen:
        seen.add(match_id)
        match_ids.append(match_id)

print("=" * 80)
print("MATCHES GESAMT:", len(match_ids))
print("BEREITS VERARBEITET:", len(results))
print("NOCH OFFEN:", len(match_ids) - len(results))
print("=" * 80)

# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/153.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
})

# ============================================================
# MATCH-SEITEN ABRUFEN
# ============================================================

for i, match_id in enumerate(match_ids, 1):

    # Bereits vorhanden -> überspringen
    if match_id in results:
        print(
            f"[{i}/{len(match_ids)}] "
            f"{match_id} -> bereits vorhanden"
        )
        continue

    game = next(
        g for g in games
        if str(g["matchId"]) == match_id
    )

    slug = game["matchSlug"]

    match_url = f"https://rft.gg/match/{match_id}-{slug}"

    success = False

    # Bis zu 5 Versuche
    for attempt in range(1, 6):

        try:
            r = session.get(
                match_url,
                timeout=30
            )

            # ------------------------------------------------
            # ERFOLG
            # ------------------------------------------------

            if r.status_code == 200:

                match_html = r.text

                # eventId
                event_ids = []

                event_ids += re.findall(
                    r'\\"eventId\\":(\d+)',
                    match_html
                )

                event_ids += re.findall(
                    r'"eventId":(\d+)',
                    match_html
                )

                event_id = (
                    event_ids[0]
                    if event_ids
                    else None
                )

                # Titel
                title_match = re.search(
                    r'<title>(.*?)</title>',
                    match_html,
                    re.DOTALL
                )

                title = (
                    title_match.group(1).strip()
                    if title_match
                    else None
                )

                results[match_id] = {
                    "matchId": match_id,
                    "slug": slug,
                    "eventId": event_id,
                    "title": title
                }

                print(
                    f"[{i}/{len(match_ids)}] "
                    f"{match_id} -> eventId={event_id}"
                )

                # Sofort speichern
                with open(
                    "match_events.json",
                    "w",
                    encoding="utf-8"
                ) as f:
                    json.dump(
                        results,
                        f,
                        ensure_ascii=False,
                        indent=2
                    )

                success = True

                # Längere Pause zwischen Requests
                time.sleep(random.uniform(2.0, 4.0))

                break

            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            elif r.status_code == 429:

                wait_time = 30 * attempt

                print(
                    f"[{i}/{len(match_ids)}] "
                    f"{match_id} -> HTTP 429 "
                    f"(Versuch {attempt}/5)"
                )

                print(
                    f"    Warte {wait_time} Sekunden..."
                )

                time.sleep(wait_time)

            # ------------------------------------------------
            # ANDERE FEHLER
            # ------------------------------------------------

            else:

                print(
                    f"[{i}/{len(match_ids)}] "
                    f"{match_id} -> HTTP {r.status_code}"
                )

                time.sleep(5)

        except Exception as e:

            print(
                f"[{i}/{len(match_ids)}] "
                f"{match_id} -> ERROR: {e}"
            )

            time.sleep(10)

    # ========================================================
    # NACH 5 FEHLGESCHLAGENEN VERSUCHEN
    # ========================================================

    if not success:

        print(
            f"    -> Übersprungen nach 5 Versuchen: "
            f"{match_id}"
        )

        # Nicht in results speichern!
        # Dadurch kann der nächste Lauf dort weitermachen.

# ============================================================
# ENDE
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print("Matches gesamt:", len(match_ids))
print("Erfolgreich:", len(results))
print("Noch offen:", len(match_ids) - len(results))
print("Datei: match_events.json")

MATCHES GESAMT: 41
BEREITS VERARBEITET: 53
NOCH OFFEN: -12
[1/41] 9503 -> bereits vorhanden
[2/41] 9523 -> bereits vorhanden
[3/41] 9555 -> bereits vorhanden
[4/41] 9567 -> bereits vorhanden
[5/41] 9647 -> bereits vorhanden
[6/41] 12540 -> bereits vorhanden
[7/41] 12547 -> bereits vorhanden
[8/41] 12551 -> bereits vorhanden
[9/41] 12556 -> bereits vorhanden
[10/41] 12564 -> bereits vorhanden
[11/41] 12567 -> bereits vorhanden
[12/41] 12572 -> bereits vorhanden
[13/41] 12576 -> bereits vorhanden
[14/41] 12582 -> bereits vorhanden
[15/41] 12586 -> bereits vorhanden
[16/41] 12591 -> bereits vorhanden
[17/41] 12599 -> bereits vorhanden
[18/41] 12604 -> bereits vorhanden
[19/41] 12608 -> bereits vorhanden
[20/41] 12612 -> bereits vorhanden
[21/41] 12617 -> bereits vorhanden
[22/41] 12620 -> bereits vorhanden
[23/41] 12628 -> bereits vorhanden
[24/41] 68800 -> bereits vorhanden
[25/41] 68844 -> bereits vorhanden
[26/41] 68849 -> bereits vorhanden
[27/41] 68853 -> bereits vorhanden
[28/41] 68

In [12]:
import requests
import json
import re
from collections import OrderedDict

# ============================================================
# JAHR EINSTELLEN
# ============================================================

YEAR = 2025

url = f"https://rft.gg/player/16-zeus?year={YEAR}"

response = requests.get(
    url,
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=20
)

print("Status:", response.status_code)
print("HTML Länge:", len(response.text))

html = response.text


# ============================================================
# PERFORMANCES FINDEN
# ============================================================

marker = '\\"performances\\":['
position = html.find(marker)

print("Performances Position:", position)

if position == -1:
    raise Exception("Performances-Array nicht gefunden.")


# ============================================================
# ARRAY EXTRAHIEREN
# ============================================================

start = position + len('\\"performances\\":')

depth = 0
in_string = False
escape = False
end = None

for i in range(start, len(html)):

    c = html[i]

    if escape:
        escape = False
        continue

    if c == "\\":
        escape = True
        continue

    if c == '"':
        in_string = not in_string
        continue

    if in_string:
        continue

    if c == "[":
        depth += 1

    elif c == "]":
        depth -= 1

        if depth == 0:
            end = i + 1
            break

if end is None:
    raise Exception("Ende des Performances-Arrays nicht gefunden.")

array_text = html[start:end]

print("Array-Länge:", len(array_text))


# ============================================================
# ESCAPING ENTFERNEN + JSON
# ============================================================

array_text = array_text.replace('\\"', '"')

games = json.loads(array_text)


# ============================================================
# AUSGABE
# ============================================================

print()
print("=" * 80)
print(f"GAMES {YEAR}:", len(games))
print("=" * 80)

if games:
    print("\nErstes Game:")
    print(json.dumps(games[0], indent=2, ensure_ascii=False))

    print("\nLetztes Game:")
    print(json.dumps(games[-1], indent=2, ensure_ascii=False))


# ============================================================
# UNIQUE MATCH IDS
# ============================================================

match_ids = list(OrderedDict.fromkeys(
    game["matchId"] for game in games
))

print()
print("=" * 80)
print("UNIQUE MATCH IDs:", len(match_ids))
print("=" * 80)

print(match_ids)


# ============================================================
# SPEICHERN
# ============================================================

filename = f"games_{YEAR}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(games, f, indent=2, ensure_ascii=False)

print()
print("=" * 80)
print("FERTIG")
print("=" * 80)
print("Gespeichert als:", filename)

Status: 200
HTML Länge: 652830
Performances Position: 551559
Array-Länge: 89972

GAMES 2025: 169

Erstes Game:
{
  "gameId": 5913248,
  "matchId": 14015,
  "matchSlug": "gen.g-esports-vs-hanwha-life-esports-2025-01-17",
  "gameDate": "$D2025-01-17T08:08:00.000Z",
  "opponentTeamId": 8768,
  "opponentTeamName": "Gen.G Esports",
  "opponentTeamShortName": "GEN",
  "opponentTeamSlug": "gen.g-esports",
  "opponentTeamImageUrl": "https://uploads.rft.gg/images/esport/teams/gen-g-esports-2025.webp",
  "rftRating": 51.4,
  "kda": 0.3333333333333333,
  "dpm": 366.7961674545182,
  "gpm": 333.0936326910936,
  "csm": 7.636986301369864,
  "vision": 1.096094749387086,
  "win": false,
  "role": "TOP"
}

Letztes Game:
{
  "gameId": 156267,
  "matchId": 66884,
  "matchSlug": "hanwha-life-esports-vs-t1-2025-12-14",
  "gameDate": "$D2025-12-14T11:40:00.000Z",
  "opponentTeamId": 272,
  "opponentTeamName": "T1",
  "opponentTeamShortName": "T1",
  "opponentTeamSlug": "t1",
  "opponentTeamImageUrl": "https:

In [13]:
import requests
import re
import json
import time

# ============================================================
# 1. 2025-GAMES LADEN
# ============================================================

with open("games_2025.json", "r", encoding="utf-8") as f:
    games_2025 = json.load(f)

# Eindeutige Matches
match_data = {}

for game in games_2025:
    match_id = game["matchId"]

    if match_id not in match_data:
        match_data[match_id] = {
            "matchId": match_id,
            "slug": game["matchSlug"]
        }

print("=" * 80)
print("MATCHES GESAMT:", len(match_data))
print("=" * 80)


# ============================================================
# 2. BEREITS GESPEICHERTE MATCH-EVENTS LADEN
# ============================================================

try:
    with open("match_events.json", "r", encoding="utf-8") as f:
        results = json.load(f)

    print("Bereits vorhandene Match-Daten:", len(results))

except FileNotFoundError:
    results = {}
    print("Keine vorhandene match_events.json gefunden.")


# ============================================================
# 3. MATCH-SEITEN ABRUFEN
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})

match_list = list(match_data.items())

for i, (match_id, data) in enumerate(match_list, 1):

    match_id_str = str(match_id)

    # Bereits vorhanden?
    if match_id_str in results:
        print(
            f"[{i}/{len(match_list)}] "
            f"{match_id} -> bereits vorhanden"
        )
        continue

    slug = data["slug"]

    match_url = f"https://rft.gg/match/{match_id}-{slug}"

    try:

        r = session.get(
            match_url,
            timeout=20
        )

        if r.status_code != 200:

            print(
                f"[{i}/{len(match_list)}] "
                f"{match_id} -> HTTP {r.status_code}"
            )

            # Bei Rate Limit stoppen
            if r.status_code == 429:
                print("\nHTTP 429 -> pausieren.")
                break

            continue

        match_html = r.text

        # ----------------------------------------------------
        # EVENT ID
        # ----------------------------------------------------

        event_ids = re.findall(
            r'\\"eventId\\":(\d+)',
            match_html
        )

        if not event_ids:

            event_ids = re.findall(
                r'"eventId":(\d+)',
                match_html
            )

        event_id = event_ids[0] if event_ids else None


        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        title_match = re.search(
            r'<title>(.*?)</title>',
            match_html,
            re.DOTALL
        )

        title = (
            title_match.group(1).strip()
            if title_match
            else None
        )


        # ----------------------------------------------------
        # SPEICHERN
        # ----------------------------------------------------

        results[match_id_str] = {
            "matchId": match_id,
            "slug": slug,
            "eventId": event_id,
            "title": title
        }

        print(
            f"[{i}/{len(match_list)}] "
            f"{match_id} -> eventId={event_id}"
        )

        # kleine Pause gegen Rate Limit
        time.sleep(0.4)


    except Exception as e:

        print(
            f"[{i}/{len(match_list)}] "
            f"{match_id} -> ERROR: {e}"
        )


# ============================================================
# 4. ZWISCHENDURCH SPEICHERN
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)
print("Match-Daten insgesamt:", len(results))
print("Datei: match_events.json")

MATCHES GESAMT: 66
Bereits vorhandene Match-Daten: 53
[1/66] 14015 -> eventId=1880
[2/66] 14019 -> eventId=1880
[3/66] 14024 -> eventId=1880
[4/66] 14027 -> eventId=1880
[5/66] 14032 -> eventId=1880
[6/66] 14039 -> eventId=1880
[7/66] 14042 -> eventId=1880
[8/66] 14044 -> eventId=1880
[9/66] 14045 -> eventId=1880
[10/66] 14048 -> eventId=1880
[11/66] 12707 -> eventId=1880
[12/66] 12710 -> eventId=1880
[13/66] 12713 -> eventId=1880
[14/66] 12715 -> eventId=1880
[15/66] 12717 -> eventId=1880
[16/66] 12718 -> eventId=1880
[17/66] 14054 -> eventId=1880
[18/66] 14058 -> eventId=1880
[19/66] 14064 -> eventId=1880
[20/66] 14070 -> eventId=1880
[21/66] 14076 -> eventId=1880
[22/66] 14081 -> eventId=1880
[23/66] 14087 -> eventId=1880
[24/66] 14093 -> eventId=1880
[25/66] 14095 -> eventId=1880
[26/66] 14099 -> eventId=1880
[27/66] 14109 -> eventId=1880
[28/66] 14113 -> eventId=1880
[29/66] 14116 -> eventId=1880
[30/66] 14122 -> eventId=1880
[31/66] 14124 -> eventId=1880
[32/66] 14130 -> eventId=

In [2]:
import json

# ============================================================
# 1. DATEIEN LADEN
# ============================================================

with open("games_2025.json", "r", encoding="utf-8") as f:
    games_2025 = json.load(f)

with open("match_events.json", "r", encoding="utf-8") as f:
    match_events = json.load(f)


# ============================================================
# 2. ERLAUBTE EVENTS 2025
# ============================================================

allowed_events = {
    "LCK 2025 Cup",
    "LCK 2025 Season",
    "LCK 2025 Road to MSI",
    "First Stand 2025",
    "MSI 2025",
    "World Championship 2025"
}


# ============================================================
# 3. EVENTNAMEN AUS MATCH-TITEL EXTRAHIEREN
# ============================================================

def get_event_name(title):

    if not title:
        return None

    parts = [x.strip() for x in title.split("|")]

    if len(parts) >= 2:
        return parts[1]

    return None


# ============================================================
# 4. GAMES FILTERN
# ============================================================

filtered_games = []
excluded_games = []

for game in games_2025:

    match_id = str(game["matchId"])

    match_data = match_events.get(match_id)

    if not match_data:
        print(
            f"WARNUNG: Match {match_id} "
            f"nicht in match_events.json"
        )
        continue

    event = get_event_name(
        match_data.get("title")
    )

    game_copy = game.copy()
    game_copy["event"] = event

    if event in allowed_events:
        filtered_games.append(game_copy)
    else:
        excluded_games.append(game_copy)


# ============================================================
# 5. AUSGABE
# ============================================================

from collections import Counter

print("=" * 80)
print("ERLAUBTE GAMES 2025:", len(filtered_games))
print("=" * 80)

for event, count in Counter(
    g["event"] for g in filtered_games
).items():
    print(f"{event:<40} {count} Games")


print()
print("=" * 80)
print("AUSGESCHLOSSENE GAMES 2025:", len(excluded_games))
print("=" * 80)

for event, count in Counter(
    g["event"] for g in excluded_games
).items():
    print(f"{event:<40} {count} Games")


# ============================================================
# 6. SPEICHERN
# ============================================================

with open(
    "filtered_games_2025.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        filtered_games,
        f,
        indent=2,
        ensure_ascii=False
    )

print()
print("=" * 80)
print("FERTIG")
print("=" * 80)
print("Gespeichert als: filtered_games_2025.json")

ERLAUBTE GAMES 2025: 147
LCK 2025 Cup                             33 Games
First Stand 2025                         17 Games
LCK 2025 Season                          80 Games
LCK 2025 Road to MSI                     8 Games
World Championship 2025                  9 Games

AUSGESCHLOSSENE GAMES 2025: 22
Esports World Cup 2025                   5 Games
KeSPA Cup 2025                           17 Games

FERTIG
Gespeichert als: filtered_games_2025.json


In [7]:
import requests
import json
import time
from collections import OrderedDict

# ============================================================
# 1. GAMES AUS DEN JAHREN LADEN
# ============================================================

years = [2024, 2023, 2022]

all_games = []

for year in years:

    filename = f"filtered_games_{year}.json"

    try:
        with open(filename, "r", encoding="utf-8") as f:
            games = json.load(f)

        print(f"{year}: {len(games)} Games geladen")

        all_games.extend(games)

    except FileNotFoundError:
        print(f"WARNUNG: {filename} nicht gefunden")


# ============================================================
# 2. MATCH IDS SAMMELN
# ============================================================

match_ids = list(OrderedDict.fromkeys(
    game["matchId"]
    for game in all_games
))

print()
print("=" * 80)
print("ALTE MATCHES GESAMT:", len(match_ids))
print("=" * 80)


# ============================================================
# 3. BEREITS GESPEICHERTE MATCH-EVENTS LADEN
# ============================================================

try:

    with open("match_events.json", "r", encoding="utf-8") as f:
        results = json.load(f)

except FileNotFoundError:

    results = {}

print("Bereits gespeichert:", len(results))


# ============================================================
# 4. NUR FEHLENDE MATCHES ABFRAGEN
# ============================================================

missing = [
    match_id
    for match_id in match_ids
    if str(match_id) not in results
]

print("Noch abzufragen:", len(missing))

print()
print("=" * 80)


session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ============================================================
# 5. MATCH-SEITEN ABFRAGEN
# ============================================================

for i, match_id in enumerate(missing, 1):

    # Game mit dieser Match-ID finden
    game = next(
        g for g in all_games
        if g["matchId"] == match_id
    )

    slug = game["matchSlug"]

    url = f"https://rft.gg/match/{match_id}-{slug}"

    try:

        response = session.get(
            url,
            timeout=30
        )

        if response.status_code == 429:

            print()
            print("HTTP 429 - Rate Limit erreicht.")
            print("Warte 30 Sekunden...")
            time.sleep(30)

            response = session.get(
                url,
                timeout=30
            )

        if response.status_code != 200:

            print(
                f"[{i}/{len(missing)}] "
                f"{match_id} -> HTTP {response.status_code}"
            )

            continue

        html = response.text

        # ----------------------------------------------------
        # eventId
        # ----------------------------------------------------

        event_id = None

        marker = '"eventId":'

        pos = html.find(marker)

        if pos != -1:

            start = pos + len(marker)

            number = ""

            while start < len(html) and html[start].isdigit():

                number += html[start]

                start += 1

            if number:
                event_id = number

        # Falls escaped
        if event_id is None:

            marker = '\\"eventId\\":'

            pos = html.find(marker)

            if pos != -1:

                start = pos + len(marker)

                number = ""

                while start < len(html) and html[start].isdigit():

                    number += html[start]

                    start += 1

                if number:
                    event_id = number


        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        title = None

        title_start = html.find("<title>")

        if title_start != -1:

            title_start += len("<title>")

            title_end = html.find(
                "</title>",
                title_start
            )

            if title_end != -1:

                title = html[
                    title_start:title_end
                ].strip()


        # ----------------------------------------------------
        # SPEICHERN
        # ----------------------------------------------------

        results[str(match_id)] = {
            "matchId": match_id,
            "slug": slug,
            "eventId": event_id,
            "title": title
        }

        print(
            f"[{i}/{len(missing)}] "
            f"{match_id} -> eventId={event_id} | {title}"
        )

        # Kleine Pause
        time.sleep(0.5)


    except Exception as e:

        print(
            f"[{i}/{len(missing)}] "
            f"{match_id} -> ERROR: {e}"
        )


# ============================================================
# 6. GESAMTE DATEI SPEICHERN
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print("Match-Events insgesamt:", len(results))
print("Datei: match_events.json")

2024: 0 Games geladen
2023: 0 Games geladen
2022: 0 Games geladen

ALTE MATCHES GESAMT: 0
Bereits gespeichert: 119
Noch abzufragen: 0


FERTIG
Match-Events insgesamt: 119
Datei: match_events.json


In [8]:
import requests
import json
import time
from collections import OrderedDict

# ============================================================
# EINSTELLUNGEN
# ============================================================

PLAYER_URL = "https://rft.gg/player/16-zeus"
YEARS = [2024, 2023, 2022]

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ============================================================
# PERFORMANCE-DATEN HOLEN
# ============================================================

def get_games(year):

    url = f"{PLAYER_URL}?year={year}"

    response = session.get(url, timeout=30)

    print(f"{year}: Status {response.status_code}")
    print(f"{year}: HTML Länge {len(response.text)}")

    html = response.text

    marker = '\\"performances\\":['
    position = html.find(marker)

    print(f"{year}: Performances Position {position}")

    if position == -1:
        return []

    start = position + len('\\"performances\\":')

    depth = 0
    in_string = False
    escape = False
    end = None

    for i in range(start, len(html)):

        c = html[i]

        if escape:
            escape = False
            continue

        if c == "\\":
            escape = True
            continue

        if c == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if c == "[":
            depth += 1

        elif c == "]":

            depth -= 1

            if depth == 0:
                end = i + 1
                break

    if end is None:
        return []

    array_text = html[start:end]
    array_text = array_text.replace('\\"', '"')

    games = json.loads(array_text)

    return games


# ============================================================
# 1. JAHRE HOLEN UND SPEICHERN
# ============================================================

all_games = []

for year in YEARS:

    print()
    print("=" * 80)
    print("JAHR", year)
    print("=" * 80)

    games = get_games(year)

    print("Games:", len(games))

    # DIREKT SPEICHERN
    with open(
        f"games_{year}.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            games,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(f"Gespeichert: games_{year}.json")

    all_games.extend(games)

    time.sleep(2)


# ============================================================
# 2. EINDEUTIGE MATCH IDS
# ============================================================

match_ids = list(OrderedDict.fromkeys(
    game["matchId"]
    for game in all_games
))

print()
print("=" * 80)
print("GAMES GESAMT:", len(all_games))
print("UNIQUE MATCH IDS:", len(match_ids))
print("=" * 80)


# ============================================================
# 3. BISHERIGE MATCH-EVENTS LADEN
# ============================================================

try:

    with open(
        "match_events.json",
        "r",
        encoding="utf-8"
    ) as f:

        results = json.load(f)

except FileNotFoundError:

    results = {}


print("Bereits gespeicherte Match-Events:", len(results))


# ============================================================
# 4. NUR NEUE MATCHES ABFRAGEN
# ============================================================

missing = [
    match_id
    for match_id in match_ids
    if str(match_id) not in results
]

print("Neue Matches:", len(missing))

print()


# ============================================================
# 5. MATCH-SEITEN ABFRAGEN
# ============================================================

for i, match_id in enumerate(missing, 1):

    game = next(
        g for g in all_games
        if g["matchId"] == match_id
    )

    slug = game["matchSlug"]

    url = f"https://rft.gg/match/{match_id}-{slug}"

    try:

        response = session.get(
            url,
            timeout=30
        )

        # Rate Limit
        if response.status_code == 429:

            print(
                f"[{i}/{len(missing)}] "
                f"{match_id} -> 429, warte 30 Sekunden..."
            )

            time.sleep(30)

            response = session.get(
                url,
                timeout=30
            )

        if response.status_code != 200:

            print(
                f"[{i}/{len(missing)}] "
                f"{match_id} -> HTTP {response.status_code}"
            )

            continue


        html = response.text


        # ----------------------------------------------------
        # EVENT ID
        # ----------------------------------------------------

        event_id = None

        for marker in [
            '"eventId":',
            '\\"eventId\\":'
        ]:

            position = html.find(marker)

            if position != -1:

                start = position + len(marker)

                number = ""

                while (
                    start < len(html)
                    and html[start].isdigit()
                ):

                    number += html[start]
                    start += 1

                if number:

                    event_id = number
                    break


        # ----------------------------------------------------
        # TITLE
        # ----------------------------------------------------

        title = None

        start = html.find("<title>")

        if start != -1:

            start += len("<title>")

            end = html.find(
                "</title>",
                start
            )

            if end != -1:

                title = html[start:end].strip()


        # ----------------------------------------------------
        # SPEICHERN
        # ----------------------------------------------------

        results[str(match_id)] = {
            "matchId": match_id,
            "slug": slug,
            "eventId": event_id,
            "title": title
        }

        print(
            f"[{i}/{len(missing)}] "
            f"{match_id} -> {event_id} | {title}"
        )

        # Nicht zu schnell
        time.sleep(0.6)


    except Exception as e:

        print(
            f"[{i}/{len(missing)}] "
            f"{match_id} -> ERROR: {e}"
        )


# ============================================================
# 6. MATCH-EVENTS SPEICHERN
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print("Match-Events insgesamt:", len(results))
print("Datei: match_events.json")


JAHR 2024
2024: Status 200
2024: HTML Länge 667739
2024: Performances Position 564444
Games: 176
Gespeichert: games_2024.json

JAHR 2023
2023: Status 200
2023: HTML Länge 635035
2023: Performances Position 538065
Games: 164
Gespeichert: games_2023.json

JAHR 2022
2022: Status 200
2022: HTML Länge 593786
2022: Performances Position 507150
Games: 144
Gespeichert: games_2022.json

GAMES GESAMT: 484
UNIQUE MATCH IDS: 192
Bereits gespeicherte Match-Events: 119
Neue Matches: 192

[1/192] 62039 -> 1880 | Gen.G Esports vs T1 | LCK 2024 Spring | January 17th, 2024 | RFT.GG
[2/192] 62043 -> 1880 | T1 vs Freecs | LCK 2024 Spring | January 19th, 2024 | RFT.GG
[3/192] 62052 -> 1880 | KT Rolster vs T1 | LCK 2024 Spring | January 26th, 2024 | RFT.GG
[4/192] 62057 -> 1880 | Hanwha Life Esports vs T1 | LCK 2024 Spring | January 28th, 2024 | RFT.GG
[5/192] 62060 -> 1426 | T1 vs DRX | LCK 2024 Spring | February 1st, 2024 | RFT.GG
[6/192] 62065 -> 1880 | BRION vs T1 | LCK 2024 Spring | February 3rd, 2024

In [9]:
import requests
import json
import time

# ============================================================
# MATCH-EVENTS LADEN
# ============================================================

with open("match_events.json", "r", encoding="utf-8") as f:
    results = json.load(f)

# ============================================================
# FEHLENDE MATCHES
# ============================================================

missing_ids = [
    53619,
    53625,
    53627,
    55572,
    55576,
    55579,
    55584,
    55588,
    55594,
    55602,
    55606,
    55608,
    55613,
    55621,
    55625,
    55632,
    55635,
    55639,
    55643
]

print("=" * 80)
print("FEHLENDE MATCHES:", len(missing_ids))
print("=" * 80)


# ============================================================
# SLUGS AUS DEN GAMES HOLEN
# ============================================================

all_games = []

for year in [2024, 2023, 2022]:

    try:
        with open(
            f"games_{year}.json",
            "r",
            encoding="utf-8"
        ) as f:

            all_games.extend(json.load(f))

    except FileNotFoundError:
        pass


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ============================================================
# MATCHES ABARBEITEN
# ============================================================

for i, match_id in enumerate(missing_ids, 1):

    game = next(
        (
            g for g in all_games
            if g["matchId"] == match_id
        ),
        None
    )

    if game is None:

        print(
            f"[{i}/{len(missing_ids)}] "
            f"{match_id} -> Game nicht gefunden"
        )

        continue

    slug = game["matchSlug"]

    url = f"https://rft.gg/match/{match_id}-{slug}"

    success = False

    # --------------------------------------------------------
    # Bis zu 5 Versuche
    # --------------------------------------------------------

    for attempt in range(1, 6):

        try:

            response = session.get(
                url,
                timeout=30
            )

            # ------------------------------------------------
            # ERFOLG
            # ------------------------------------------------

            if response.status_code == 200:

                html = response.text

                # eventId
                event_id = None

                for marker in [
                    '"eventId":',
                    '\\"eventId\\":'
                ]:

                    pos = html.find(marker)

                    if pos != -1:

                        start = pos + len(marker)

                        number = ""

                        while (
                            start < len(html)
                            and html[start].isdigit()
                        ):

                            number += html[start]
                            start += 1

                        if number:
                            event_id = number
                            break


                # Titel
                title = None

                start = html.find("<title>")

                if start != -1:

                    start += len("<title>")

                    end = html.find(
                        "</title>",
                        start
                    )

                    if end != -1:
                        title = html[start:end].strip()


                # Speichern
                results[str(match_id)] = {
                    "matchId": match_id,
                    "slug": slug,
                    "eventId": event_id,
                    "title": title
                }

                print(
                    f"[{i}/{len(missing_ids)}] "
                    f"{match_id} -> {event_id} | {title}"
                )

                success = True

                # Direkt speichern, falls wieder 429 kommt
                with open(
                    "match_events.json",
                    "w",
                    encoding="utf-8"
                ) as f:

                    json.dump(
                        results,
                        f,
                        indent=2,
                        ensure_ascii=False
                    )

                break


            # ------------------------------------------------
            # RATE LIMIT
            # ------------------------------------------------

            elif response.status_code == 429:

                wait = 30 * attempt

                print(
                    f"[{i}/{len(missing_ids)}] "
                    f"{match_id} -> 429 "
                    f"(Versuch {attempt}/5), "
                    f"warte {wait}s..."
                )

                time.sleep(wait)


            else:

                print(
                    f"[{i}/{len(missing_ids)}] "
                    f"{match_id} -> HTTP "
                    f"{response.status_code}"
                )

                break


        except Exception as e:

            print(
                f"[{i}/{len(missing_ids)}] "
                f"{match_id} -> ERROR: {e}"
            )

            time.sleep(10)


    if not success:

        print(
            f"!!! {match_id} konnte nicht geladen werden"
        )

    # normale Pause zwischen erfolgreichen Requests
    time.sleep(2)


# ============================================================
# ABSCHLUSS
# ============================================================

with open(
    "match_events.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print("Match-Events insgesamt:", len(results))

print()
print("Noch fehlend:")

for match_id in missing_ids:

    if str(match_id) not in results:
        print(match_id)

FEHLENDE MATCHES: 19
[1/19] 53619 -> 1880 | Gen.G Esports vs T1 | MSI 2023 | May 13th, 2023 | RFT.GG
[2/19] 53625 -> 1880 | T1 vs JD Gaming | MSI 2023 | May 18th, 2023 | RFT.GG
[3/19] 53627 -> 1880 | T1 vs Bilibili Gaming | MSI 2023 | May 20th, 2023 | RFT.GG
[4/19] 55572 -> 1880 | T1 vs Hanwha Life Esports | LCK 2023 Summer | June 9th, 2023 | RFT.GG
[5/19] 55576 -> 1880 | Gen.G Esports vs T1 | LCK 2023 Summer | June 11th, 2023 | RFT.GG
[6/19] 55579 -> 1880 | T1 vs SANDBOX Gaming | LCK 2023 Summer | June 15th, 2023 | RFT.GG
[7/19] 55584 -> 1880 | BRION vs T1 | LCK 2023 Summer | June 17th, 2023 | RFT.GG
[8/19] 55588 -> 1880 | Freecs vs T1 | LCK 2023 Summer | June 21st, 2023 | RFT.GG
[9/19] 55594 -> 1880 | T1 vs KT Rolster | LCK 2023 Summer | June 24th, 2023 | RFT.GG
[10/19] 55602 -> 1880 | T1 vs Dplus | LCK 2023 Summer | June 30th, 2023 | RFT.GG
[11/19] 55606 -> 1880 | Nongshim RedForce vs T1 | LCK 2023 Summer | July 2nd, 2023 | RFT.GG
[12/19] 55608 -> 1880 | DRX vs T1 | LCK 2023 Summer 

In [1]:
import requests
import re
import json
import time
from collections import Counter

# ============================================================
# EINSTELLUNGEN
# ============================================================

PLAYER_ID = 16
YEARS = [2022, 2023, 2024, 2025, 2026]

# Bereits vorhandene Match -> Event Daten
with open("match_events.json", "r", encoding="utf-8") as f:
    match_events = json.load(f)

print("=" * 80)
print("MATCH-EVENTS GELADEN:", len(match_events))
print("=" * 80)


# ============================================================
# ERLAUBTE EVENTS
# ============================================================

ALLOWED_KEYWORDS = [
    # 2022
    "LCK 2022 Spring",
    "LCK 2022 Summer",
    "LCK 2022 Regional Finals",
    "MSI 2022",
    "World Championship 2022",

    # 2023
    "LCK 2023 Spring",
    "LCK 2023 Summer",
    "LCK 2023 Regional Finals",
    "MSI 2023",
    "World Championship 2023",

    # 2024
    "LCK 2024 Spring",
    "LCK 2024 Summer",
    "LCK 2024 Regional Finals",
    "MSI 2024",
    "World Championship 2024",

    # 2025
    "LCK 2025 Cup",
    "LCK 2025 Season",
    "LCK 2025 Road to MSI",
    "First Stand 2025",
    "MSI 2025",
    "World Championship 2025",

    # 2026
    "LCK 2026 Cup",
    "LCK 2026 Season",
    "LCK 2026 Road to MSI",
    "First Stand 2026",
    "MSI 2026",
    "World Championship 2026",
]


# ============================================================
# PERFORMANCE-ARRAY AUS PLAYER-SEITE
# ============================================================

def load_games(year):

    url = f"https://rft.gg/player/{PLAYER_ID}-zeus?year={year}"

    print()
    print("#" * 80)
    print(f"# JAHR {year}")
    print("#" * 80)
    print()
    print("URL:", url)

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=20
    )

    print("Status:", response.status_code)
    print("HTML Länge:", len(response.text))

    html = response.text

    marker = '\\"performances\\":['
    position = html.find(marker)

    print("Performances Position:", position)

    if position == -1:
        raise Exception(
            f"Performances-Array für {year} nicht gefunden."
        )

    start = position + len('\\"performances\\":')

    depth = 0
    in_string = False
    escape = False
    end = None

    for i in range(start, len(html)):

        c = html[i]

        if escape:
            escape = False
            continue

        if c == "\\":
            escape = True
            continue

        if c == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if c == "[":
            depth += 1

        elif c == "]":
            depth -= 1

            if depth == 0:
                end = i + 1
                break

    if end is None:
        raise Exception(
            f"Ende des Performances-Arrays für {year} nicht gefunden."
        )

    array_text = html[start:end]

    print("Array-Länge:", len(array_text))

    array_text = array_text.replace('\\"', '"')

    games = json.loads(array_text)

    print()
    print("=" * 80)
    print(f"GESAMTE GAMES {year}:", len(games))
    print("=" * 80)

    return games


# ============================================================
# EVENT-FILTER
# ============================================================

def get_event(match_id):

    data = match_events.get(str(match_id))

    if not data:
        return None

    return data.get("title")


def is_allowed(event):

    if not event:
        return False

    for keyword in ALLOWED_KEYWORDS:
        if keyword.lower() in event.lower():
            return True

    return False


# ============================================================
# ALLE JAHRE VERARBEITEN
# ============================================================

all_filtered_games = []

year_counts = {}

for year in YEARS:

    try:
        games = load_games(year)

    except Exception as e:
        print("FEHLER:", e)
        continue

    filtered = []

    missing_event = 0

    for game in games:

        event = get_event(game["matchId"])

        if event is None:
            missing_event += 1
            continue

        if is_allowed(event):

            game_copy = game.copy()
            game_copy["event"] = event
            game_copy["eventId"] = match_events[
                str(game["matchId"])
            ].get("eventId")

            filtered.append(game_copy)

    year_counts[year] = len(filtered)

    all_filtered_games.extend(filtered)

    print()
    print("=" * 80)
    print(f"ERLAUBTE GAMES {year}:", len(filtered))
    print("=" * 80)

    if missing_event:
        print(
            "Games ohne Match-Event:",
            missing_event
        )

    # Einzeldatei speichern
    with open(
        f"filtered_games_{year}.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            filtered,
            f,
            indent=2,
            ensure_ascii=False
        )

    print(
        f"Gespeichert als: filtered_games_{year}.json"
    )


# ============================================================
# GESAMTDATEI
# ============================================================

with open(
    "all_filtered_games_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        all_filtered_games,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# ÜBERSICHT
# ============================================================

print()
print()
print("=" * 80)
print("ALLE JAHRE FERTIG")
print("=" * 80)

for year in YEARS:
    print(
        f"{year}: "
        f"{year_counts.get(year, 0)} erlaubte Games"
    )

print()
print(
    "GESAMT:",
    len(all_filtered_games),
    "Games"
)

print()
print(
    "Gespeichert als:",
    "all_filtered_games_2022_2026.json"
)


# ============================================================
# EVENT-ÜBERSICHT
# ============================================================

print()
print("=" * 80)
print("EVENTS GESAMT")
print("=" * 80)

event_counts = Counter(
    game["event"]
    for game in all_filtered_games
)

for event, count in event_counts.items():
    print(
        f"{event:<50} {count} Games"
    )

MATCH-EVENTS GELADEN: 311

################################################################################
# JAHR 2022
################################################################################

URL: https://rft.gg/player/16-zeus?year=2022
Status: 200
HTML Länge: 593768
Performances Position: 507119
Array-Länge: 75350

GESAMTE GAMES 2022: 144

ERLAUBTE GAMES 2022: 144
Gespeichert als: filtered_games_2022.json

################################################################################
# JAHR 2023
################################################################################

URL: https://rft.gg/player/16-zeus?year=2023
Status: 200
HTML Länge: 635145
Performances Position: 538163
Array-Länge: 85683

GESAMTE GAMES 2023: 164

ERLAUBTE GAMES 2023: 152
Gespeichert als: filtered_games_2023.json

################################################################################
# JAHR 2024
################################################################################

URL: https

In [ ]:
import json
from collections import defaultdict

# ============================================================
# 1. DATEN LADEN
# ============================================================

with open(
    "all_filtered_games_2022_2026.json",
    "r",
    encoding="utf-8"
) as f:
    games = json.load(f)

print("=" * 80)
print("GAMES GELADEN:", len(games))
print("=" * 80)


# ============================================================
# 2. NACH EVENT AGGREGIEREN
# ============================================================

events = defaultdict(list)

for game in games:
    events[game["event"]].append(game)


# ============================================================
# 3. STATISTIKEN BERECHNEN
# ============================================================

event_stats = []

for event, event_games in events.items():

    games_count = len(event_games)

    wins = sum(
        1 for game in event_games
        if game["win"]
    )

    losses = games_count - wins

    avg_kda = sum(
        game["kda"]
        for game in event_games
    ) / games_count

    avg_dpm = sum(
        game["dpm"]
        for game in event_games
    ) / games_count

    avg_gpm = sum(
        game["gpm"]
        for game in event_games
    ) / games_count

    avg_csm = sum(
        game["csm"]
        for game in event_games
    ) / games_count

    avg_vision = sum(
        game["vision"]
        for game in event_games
    ) / games_count

    avg_rft = sum(
        game["rftRating"]
        for game in event_games
    ) / games_count

    event_stats.append({
        "event": event,
        "games": games_count,
        "wins": wins,
        "losses": losses,
        "winrate": wins / games_count,
        "kda": avg_kda,
        "dpm": avg_dpm,
        "gpm": avg_gpm,
        "csm": avg_csm,
        "vision": avg_vision,
        "rftRating": avg_rft
    })


# ============================================================
# 4. SORTIEREN
# ============================================================

# Nach Eventname
event_stats.sort(
    key=lambda x: x["event"]
)


# ============================================================
# 5. AUSGABE
# ============================================================

print()
print("=" * 80)
print("EVENT-STATISTIKEN")
print("=" * 80)

for stat in event_stats:

    print()
    print(stat["event"])

    print(
        f"  Games:    {stat['games']}"
    )

    print(
        f"  W/L:      "
        f"{stat['wins']}/{stat['losses']}"
    )

    print(
        f"  Winrate:  "
        f"{stat['winrate'] * 100:.2f}%"
    )

    print(
        f"  KDA:      "
        f"{stat['kda']:.3f}"
    )

    print(
        f"  DPM:      "
        f"{stat['dpm']:.2f}"
    )

    print(
        f"  GPM:      "
        f"{stat['gpm']:.2f}"
    )

    print(
        f"  CSM:      "
        f"{stat['csm']:.3f}"
    )

    print(
        f"  Vision:   "
        f"{stat['vision']:.3f}"
    )

    print(
        f"  RFT:      "
        f"{stat['rftRating']:.2f}"
    )


# ============================================================
# 6. ALS JSON SPEICHERN
# ============================================================

with open(
    "event_stats_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        event_stats,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print(
    "Events:",
    len(event_stats)
)

print(
    "Datei:",
    "event_stats_2022_2026.json"
)

In [3]:
import json
from collections import defaultdict

# ============================================================
# 1. DATEN LADEN
# ============================================================

with open(
    "all_filtered_games_2022_2026.json",
    "r",
    encoding="utf-8"
) as f:
    games = json.load(f)

print("=" * 80)
print("GAMES GELADEN:", len(games))
print("=" * 80)


# ============================================================
# 2. NACH JAHR GRUPPIEREN
# ============================================================

years = defaultdict(list)

for game in games:

    # gameDate sieht z.B. so aus:
    # $D2026-01-16T10:08:00.000Z

    date = game["gameDate"]

    year = int(
        date[2:6]
    )

    years[year].append(game)


# ============================================================
# 3. JAHRESSTATISTIKEN
# ============================================================

year_stats = []

for year in sorted(years):

    year_games = years[year]

    games_count = len(year_games)

    wins = sum(
        game["win"]
        for game in year_games
    )

    losses = games_count - wins

    avg_kda = sum(
        game["kda"]
        for game in year_games
    ) / games_count

    avg_dpm = sum(
        game["dpm"]
        for game in year_games
    ) / games_count

    avg_gpm = sum(
        game["gpm"]
        for game in year_games
    ) / games_count

    avg_csm = sum(
        game["csm"]
        for game in year_games
    ) / games_count

    avg_vision = sum(
        game["vision"]
        for game in year_games
    ) / games_count

    avg_rft = sum(
        game["rftRating"]
        for game in year_games
    ) / games_count

    year_stats.append({
        "year": year,
        "games": games_count,
        "wins": wins,
        "losses": losses,
        "winrate": wins / games_count,
        "kda": avg_kda,
        "dpm": avg_dpm,
        "gpm": avg_gpm,
        "csm": avg_csm,
        "vision": avg_vision,
        "rftRating": avg_rft
    })


# ============================================================
# 4. AUSGABE
# ============================================================

print()
print("=" * 80)
print("JAHRESSTATISTIKEN")
print("=" * 80)

for stat in year_stats:

    print()
    print(f"YEAR {stat['year']}")
    print("-" * 40)

    print(
        f"Games:    {stat['games']}"
    )

    print(
        f"W/L:      "
        f"{stat['wins']}/{stat['losses']}"
    )

    print(
        f"Winrate:  "
        f"{stat['winrate'] * 100:.2f}%"
    )

    print(
        f"KDA:      "
        f"{stat['kda']:.3f}"
    )

    print(
        f"DPM:      "
        f"{stat['dpm']:.2f}"
    )

    print(
        f"GPM:      "
        f"{stat['gpm']:.2f}"
    )

    print(
        f"CSM:      "
        f"{stat['csm']:.3f}"
    )

    print(
        f"Vision:   "
        f"{stat['vision']:.3f}"
    )

    print(
        f"RFT:      "
        f"{stat['rftRating']:.2f}"
    )


# ============================================================
# 5. SPEICHERN
# ============================================================

with open(
    "year_stats_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        year_stats,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print(
    "Datei:",
    "year_stats_2022_2026.json"
)

GAMES GELADEN: 728

JAHRESSTATISTIKEN

YEAR 2022
----------------------------------------
Games:    144
W/L:      107/37
Winrate:  74.31%
KDA:      5.193
DPM:      1633.88
GPM:      418.11
CSM:      8.144
Vision:   0.893
RFT:      77.01

YEAR 2023
----------------------------------------
Games:    152
W/L:      96/56
Winrate:  63.16%
KDA:      4.595
DPM:      513.73
GPM:      401.18
CSM:      8.363
Vision:   1.137
RFT:      69.33

YEAR 2024
----------------------------------------
Games:    166
W/L:      106/60
Winrate:  63.86%
KDA:      4.286
DPM:      534.69
GPM:      408.55
CSM:      8.303
Vision:   1.081
RFT:      68.69

YEAR 2025
----------------------------------------
Games:    147
W/L:      95/52
Winrate:  64.63%
KDA:      4.768
DPM:      640.72
GPM:      402.88
CSM:      7.936
Vision:   1.057
RFT:      70.43

YEAR 2026
----------------------------------------
Games:    119
W/L:      72/47
Winrate:  60.50%
KDA:      5.330
DPM:      655.33
GPM:      397.73
CSM:      8.237
Vision

In [1]:
import requests
import re
import json
import time
from collections import defaultdict, Counter


# ============================================================
# EINSTELLUNGEN
# ============================================================

PLAYER_ID = 911
PLAYER_SLUG = "kiin"
PLAYER_NAME = "Kiin"

YEARS = [2022, 2023, 2024, 2025, 2026]

MATCH_EVENTS_FILE = "match_events.json"


# ============================================================
# ERLAUBTE EVENTS
# ============================================================

ALLOWED_KEYWORDS = [

    # 2022
    "LCK 2022 Spring",
    "LCK 2022 Summer",
    "LCK 2022 Regional Finals",
    "MSI 2022",
    "World Championship 2022",

    # 2023
    "LCK 2023 Spring",
    "LCK 2023 Summer",
    "LCK 2023 Regional Finals",
    "MSI 2023",
    "World Championship 2023",

    # 2024
    "LCK 2024 Spring",
    "LCK 2024 Summer",
    "LCK 2024 Regional Finals",
    "MSI 2024",
    "World Championship 2024",

    # 2025
    "LCK 2025 Cup",
    "LCK 2025 Season",
    "LCK 2025 Road to MSI",
    "First Stand 2025",
    "MSI 2025",
    "World Championship 2025",

    # 2026
    "LCK 2026 Cup",
    "LCK 2026 Season",
    "LCK 2026 Road to MSI",
    "First Stand 2026",
    "MSI 2026",
    "World Championship 2026",
]


# ============================================================
# MATCH-EVENTS LADEN
# ============================================================

try:

    with open(
        MATCH_EVENTS_FILE,
        "r",
        encoding="utf-8"
    ) as f:

        match_events = json.load(f)

except FileNotFoundError:

    match_events = {}


print("=" * 80)
print("MATCH-EVENTS GELADEN:", len(match_events))
print("=" * 80)


# ============================================================
# SESSION
# ============================================================

session = requests.Session()

session.headers.update({
    "User-Agent": "Mozilla/5.0"
})


# ============================================================
# PERFORMANCE-DATEN VON SPIELERSEITE LADEN
# ============================================================

def load_games(year):

    url = (
        f"https://rft.gg/player/"
        f"{PLAYER_ID}-{PLAYER_SLUG}?year={year}"
    )

    print()
    print("#" * 80)
    print(f"# {PLAYER_NAME.upper()} – JAHR {year}")
    print("#" * 80)
    print()
    print("URL:", url)

    response = session.get(
        url,
        timeout=20
    )

    print("Status:", response.status_code)
    print("HTML Länge:", len(response.text))

    html = response.text

    marker = '\\"performances\\":['

    position = html.find(marker)

    print(
        "Performances Position:",
        position
    )

    if position == -1:
        raise Exception(
            "Performances-Array nicht gefunden."
        )

    start = position + len(
        '\\"performances\\":'
    )

    depth = 0
    in_string = False
    escape = False
    end = None

    for i in range(
        start,
        len(html)
    ):

        c = html[i]

        if escape:
            escape = False
            continue

        if c == "\\":
            escape = True
            continue

        if c == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if c == "[":
            depth += 1

        elif c == "]":

            depth -= 1

            if depth == 0:
                end = i + 1
                break

    if end is None:
        raise Exception(
            "Ende des Performances-Arrays nicht gefunden."
        )

    array_text = html[start:end]

    print(
        "Array-Länge:",
        len(array_text)
    )

    array_text = array_text.replace(
        '\\"',
        '"'
    )

    games = json.loads(
        array_text
    )

    print()
    print(
        f"GAMES {year}:",
        len(games)
    )

    return games


# ============================================================
# ALLE SPIELER-GAMES LADEN
# ============================================================

all_games = []

for year in YEARS:

    try:

        year_games = load_games(year)

        all_games.extend(
            year_games
        )

        time.sleep(0.5)

    except Exception as e:

        print(
            f"FEHLER {year}:",
            e
        )


# ============================================================
# DOPPELTE GAMES ENTFERNEN
# ============================================================

unique_games = {}

for game in all_games:

    unique_games[
        game["gameId"]
    ] = game


games = list(
    unique_games.values()
)


print()
print("=" * 80)
print(
    "GAMES INSGESAMT:",
    len(games)
)
print("=" * 80)


# ============================================================
# MATCH-IDS ERMITTELN
# ============================================================

match_ids = []

seen = set()

for game in games:

    match_id = str(
        game["matchId"]
    )

    if match_id not in seen:

        seen.add(match_id)

        match_ids.append(
            match_id
        )


print(
    "UNIQUE MATCHES:",
    len(match_ids)
)


# ============================================================
# FEHLENDE MATCH-EVENTS ABFRAGEN
# ============================================================

missing_matches = [
    match_id
    for match_id in match_ids
    if match_id not in match_events
]


print()
print("=" * 80)
print(
    "BEREITS GESPEICHERTE MATCHES:",
    len(match_events)
)

print(
    "FEHLENDE MATCHES:",
    len(missing_matches)
)
print("=" * 80)


for index, match_id in enumerate(
    missing_matches,
    1
):

    game = next(
        g for g in games
        if str(g["matchId"]) == match_id
    )

    slug = game["matchSlug"]

    url = (
        f"https://rft.gg/match/"
        f"{match_id}-{slug}"
    )

    while True:

        try:

            response = session.get(
                url,
                timeout=20
            )

            if response.status_code == 429:

                print(
                    f"[{index}/{len(missing_matches)}] "
                    f"{match_id} -> 429, "
                    f"warte 30 Sekunden..."
                )

                time.sleep(30)

                continue

            break

        except Exception as e:

            print(
                f"[{index}/{len(missing_matches)}] "
                f"{match_id} -> ERROR:",
                e
            )

            time.sleep(5)


    if response.status_code != 200:

        print(
            f"[{index}/{len(missing_matches)}] "
            f"{match_id} -> HTTP "
            f"{response.status_code}"
        )

        continue


    match_html = response.text


    # --------------------------------------------------------
    # EVENT ID
    # --------------------------------------------------------

    event_ids = re.findall(
        r'\\"eventId\\":(\d+)',
        match_html
    )

    if not event_ids:

        event_ids = re.findall(
            r'"eventId":(\d+)',
            match_html
        )


    event_id = (
        event_ids[0]
        if event_ids
        else None
    )


    # --------------------------------------------------------
    # TITEL
    # --------------------------------------------------------

    title_match = re.search(
        r'<title>(.*?)</title>',
        match_html,
        re.DOTALL
    )

    title = (
        title_match.group(1).strip()
        if title_match
        else None
    )


    match_events[match_id] = {

        "matchId": int(match_id),

        "slug": slug,

        "eventId": event_id,

        "title": title
    }


    print(
        f"[{index}/{len(missing_matches)}] "
        f"{match_id} -> "
        f"{event_id} | "
        f"{title}"
    )


    # Nicht zu schnell anfragen
    time.sleep(1)


# ============================================================
# MATCH-EVENTS SPEICHERN
# ============================================================

with open(
    MATCH_EVENTS_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        match_events,
        f,
        indent=2,
        ensure_ascii=False
    )


print()
print(
    "Match-Events gespeichert."
)


# ============================================================
# EVENTS DEN GAMES ZUORDNEN
# ============================================================

games_with_events = []

unknown_events = []

for game in games:

    match_id = str(
        game["matchId"]
    )

    data = match_events.get(
        match_id
    )

    if not data:

        unknown_events.append(
            match_id
        )

        continue


    event = data.get(
        "title"
    )


    if not event:

        unknown_events.append(
            match_id
        )

        continue


    game_copy = game.copy()

    game_copy["event"] = event

    game_copy["eventId"] = data.get(
        "eventId"
    )

    games_with_events.append(
        game_copy
    )


# ============================================================
# ERLAUBTE EVENTS FILTERN
# ============================================================

filtered_games = []

excluded_events = Counter()

for game in games_with_events:

    event = game["event"]

    allowed = any(
        keyword.lower()
        in event.lower()
        for keyword in ALLOWED_KEYWORDS
    )

    if allowed:

        filtered_games.append(
            game
        )

    else:

        excluded_events[event] += 1


# ============================================================
# EVENT-STATISTIKEN
# ============================================================

events = defaultdict(list)

for game in filtered_games:

    events[
        game["event"]
    ].append(game)


event_stats = []

for event, event_games in events.items():

    count = len(event_games)

    wins = sum(
        game["win"]
        for game in event_games
    )

    losses = count - wins

    stat = {

        "event": event,

        "games": count,

        "wins": wins,

        "losses": losses,

        "winrate": wins / count,

        "kda": sum(
            game["kda"]
            for game in event_games
        ) / count,

        "dpm": sum(
            game["dpm"]
            for game in event_games
        ) / count,

        "gpm": sum(
            game["gpm"]
            for game in event_games
        ) / count,

        "csm": sum(
            game["csm"]
            for game in event_games
        ) / count,

        "vision": sum(
            game["vision"]
            for game in event_games
        ) / count,

        "rftRating": sum(
            game["rftRating"]
            for game in event_games
        ) / count
    }

    event_stats.append(
        stat
    )


event_stats.sort(
    key=lambda x: x["event"]
)


# ============================================================
# JAHRESSTATISTIKEN
# ============================================================

years = defaultdict(list)

for game in filtered_games:

    date = game["gameDate"]

    year = int(
        date[2:6]
    )

    years[year].append(
        game
    )


year_stats = []

for year in sorted(years):

    year_games = years[year]

    count = len(year_games)

    wins = sum(
        game["win"]
        for game in year_games
    )

    losses = count - wins

    stat = {

        "year": year,

        "games": count,

        "wins": wins,

        "losses": losses,

        "winrate": wins / count,

        "kda": sum(
            game["kda"]
            for game in year_games
        ) / count,

        "dpm": sum(
            game["dpm"]
            for game in year_games
        ) / count,

        "gpm": sum(
            game["gpm"]
            for game in year_games
        ) / count,

        "csm": sum(
            game["csm"]
            for game in year_games
        ) / count,

        "vision": sum(
            game["vision"]
            for game in year_games
        ) / count,

        "rftRating": sum(
            game["rftRating"]
            for game in year_games
        ) / count
    }

    year_stats.append(
        stat
    )


# ============================================================
# DATEIEN SPEICHERN
# ============================================================

with open(
    f"{PLAYER_SLUG}_games_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        filtered_games,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    f"{PLAYER_SLUG}_event_stats_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        event_stats,
        f,
        indent=2,
        ensure_ascii=False
    )


with open(
    f"{PLAYER_SLUG}_year_stats_2022_2026.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        year_stats,
        f,
        indent=2,
        ensure_ascii=False
    )


# ============================================================
# AUSGABE
# ============================================================

print()
print()
print("=" * 80)
print("FERTIG")
print("=" * 80)

print()
print("SPIELER:", PLAYER_NAME)

print()
print("ERLAUBTE GAMES GESAMT:")
print(len(filtered_games))

print()
print("GAMES PRO JAHR:")

for stat in year_stats:

    print(
        f"{stat['year']}: "
        f"{stat['games']} Games | "
        f"{stat['wins']}W "
        f"{stat['losses']}L | "
        f"WR {stat['winrate'] * 100:.2f}% | "
        f"KDA {stat['kda']:.3f} | "
        f"DPM {stat['dpm']:.2f} | "
        f"RFT {stat['rftRating']:.2f}"
    )


print()
print("EVENTS:")

for stat in event_stats:

    print(
        f"{stat['event']:<55} "
        f"{stat['games']:>3} Games"
    )


print()
print("AUSGESCHLOSSENE EVENTS:")

for event, count in excluded_events.items():

    print(
        f"{event:<55} "
        f"{count:>3} Games"
    )


print()
print("DATEIEN:")

print(
    f"  {PLAYER_SLUG}_games_2022_2026.json"
)

print(
    f"  {PLAYER_SLUG}_event_stats_2022_2026.json"
)

print(
    f"  {PLAYER_SLUG}_year_stats_2022_2026.json"
)

MATCH-EVENTS GELADEN: 311

################################################################################
# KIIN – JAHR 2022
################################################################################

URL: https://rft.gg/player/911-kiin?year=2022
Status: 200
HTML Länge: 475947
Performances Position: 416549
Array-Länge: 48117

GAMES 2022: 92

################################################################################
# KIIN – JAHR 2023
################################################################################

URL: https://rft.gg/player/911-kiin?year=2023
Status: 200
HTML Länge: 557301
Performances Position: 478429
Array-Länge: 67580

GAMES 2023: 129

################################################################################
# KIIN – JAHR 2024
################################################################################

URL: https://rft.gg/player/911-kiin?year=2024
Status: 200
HTML Länge: 598094
Performances Position: 511147
Array-Länge: 75671

GAMES 2024: 1